### Data Loading and Cleaning

In [28]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.preprocessing import load_dermatology_data, handle_missing_age, validate_dataset, split_data

X, y = load_dermatology_data()
X = handle_missing_age(X)
validate_dataset(X, y)
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

SEED = 42

results = {}

In [29]:
X_train.shape

(219, 34)

In [30]:
X_val.shape

(73, 34)

In [31]:
X_test.shape

(74, 34)

In [32]:
y_train.shape

(219, 1)

In [33]:
y_val.shape

(73, 1)

In [34]:
y_test.shape

(74, 1)

### Baseline Model - Logistic Regression

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

log_reg_model = LogisticRegression(random_state=SEED)
log_reg_model.fit(X_train_scaled, y_train.values.ravel())

y_preds = log_reg_model.predict(X_val_scaled)

log_reg_f1_score = f1_score(y_val, y_preds, average='macro')

results['logistic_regression'] = log_reg_f1_score
print(classification_report(y_val, y_preds))
print(f"Macro F1: {log_reg_f1_score}")

              precision    recall  f1-score   support

           1       1.00      1.00      1.00        22
           2       0.92      0.92      0.92        12
           3       1.00      1.00      1.00        14
           4       0.90      0.90      0.90        10
           5       1.00      1.00      1.00        11
           6       1.00      1.00      1.00         4

    accuracy                           0.97        73
   macro avg       0.97      0.97      0.97        73
weighted avg       0.97      0.97      0.97        73

Macro F1: 0.9694444444444444


### Next Model: Random Forest

In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
rand_forest_model = RandomForestClassifier(random_state=SEED)
rand_forest_model.fit(X_train, y_train.values.ravel())

y_preds = rand_forest_model.predict(X_val)

rand_forest_f1_score = f1_score(y_val, y_preds, average='macro')

results['random_forest'] = rand_forest_f1_score
print(classification_report(y_val, y_preds))
print(f"Macro F1: {rand_forest_f1_score}")


              precision    recall  f1-score   support

           1       1.00      1.00      1.00        22
           2       1.00      1.00      1.00        12
           3       1.00      1.00      1.00        14
           4       1.00      1.00      1.00        10
           5       1.00      1.00      1.00        11
           6       1.00      1.00      1.00         4

    accuracy                           1.00        73
   macro avg       1.00      1.00      1.00        73
weighted avg       1.00      1.00      1.00        73

Macro F1: 1.0


### Next Model: XGBoost

In [64]:
import xgboost as xgb

y_train_xgb = y_train.values.ravel() - 1
y_val_xgb = y_val.values.ravel() - 1

xgb_model = xgb.XGBClassifier(
    random_state=SEED,
    objective='multi:softprob')
xgb_model.fit(X_train, y_train_xgb)

y_preds = xgb_model.predict(X_val)

xgb_f1_score = f1_score(y_val_xgb, y_preds, average='macro')

results['xgboost'] = xgb_f1_score
print(classification_report(y_val_xgb, y_preds))    
print(f"Macro F1: {xgb_f1_score}")

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        22
           1       0.92      0.92      0.92        12
           2       1.00      1.00      1.00        14
           3       0.90      0.90      0.90        10
           4       1.00      1.00      1.00        11
           5       1.00      1.00      1.00         4

    accuracy                           0.97        73
   macro avg       0.97      0.97      0.97        73
weighted avg       0.97      0.97      0.97        73

Macro F1: 0.9694444444444444
